In [1]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import seaborn as sns

OUT = "models"; CSV = f"{OUT}/csv"; VIZ = f"{OUT}/viz"

def _find_up(name, start=None):
    d = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.exists(os.path.join(d, name)):
            return d
        p = os.path.dirname(d)
        if p == d:
            raise FileNotFoundError(name)
        d = p

PROJECT = _find_up("Data Science Task Data.xlsx - Result 1.csv")
OUT = f"{PROJECT}/models"; CSV = f"{OUT}/csv"; VIZ = f"{OUT}/viz"
for d_ in (OUT, CSV, VIZ):
    os.makedirs(d_, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

mast = pd.read_csv(f"{PROJECT}/master_dataset_clusters.csv", low_memory=False,
                   parse_dates=["registration_date", "birth_date", "first_deposit_day", "closed_at"])
REF = mast["registration_date"].max()
m = mast.dropna(subset=["cluster_kmeans"]).copy()
m["cluster_kmeans"] = m["cluster_kmeans"].astype(int)
print("profiled players:", len(m), "| reference date:", REF)

m["tenure_days"] = (REF - m["registration_date"]).dt.days
m["age"] = (REF - m["birth_date"]).dt.days / 365.25
m["ttfd_days"] = (m["first_deposit_day"] - m["registration_date"]).dt.days
m["net"] = m["total_deposits"] - m["total_withdrawals"]
m["win_rate"] = (m["total_wins"] / m["total_bets"].replace(0, np.nan)) * 100
m["monthly_value"] = m["total_deposits"] / (m["tenure_days"] / 30.4375).replace(0, np.nan)
print(m["cluster_kmeans"].value_counts().sort_index().to_dict())

profiled players: 59922 | reference date: 2025-11-13 07:39:26.364000
{0: 9078, 1: 13366, 2: 10787, 3: 15122, 4: 8125, 5: 3444}


In [2]:
# ---- numeric / value profile per cluster (medians; % within cluster) ----
def row(name, s, fmt=".0f"):
    return m.groupby("cluster_kmeans").agg(v=(name, lambda x: x.median(skipna=True))).round(0)

prof = pd.DataFrame({
    "players": m.groupby("cluster_kmeans").size(),
    "share_pct": (100 * m.groupby("cluster_kmeans").size() /
                  len(m)).round(1),
    "med_tenure_days": m.groupby("cluster_kmeans")["tenure_days"].median().round(0),
    "med_deposits": m.groupby("cluster_kmeans")["total_deposits"].median().round(0),
    "med_withdrawals": m.groupby("cluster_kmeans")["total_withdrawals"].median().round(0),
    "med_bets": m.groupby("cluster_kmeans")["total_bets"].median().round(0),
    "med_logins": m.groupby("cluster_kmeans")["total_logins"].median().round(1),
    "med_net": m.groupby("cluster_kmeans")["net"].median().round(0),
    "med_win_rate_pct": m.groupby("cluster_kmeans")["win_rate"].median().round(1),
    "med_monthly_deposits": m.groupby("cluster_kmeans")["monthly_value"].median().round(0),
    "med_ttfd_days": m.groupby("cluster_kmeans")["ttfd_days"].median().round(0),
    "med_age": m.groupby("cluster_kmeans")["age"].median().round(0),
    "pct_male": (100 * m.groupby("cluster_kmeans")["gender"].
                 apply(lambda s: s.eq("Male").mean())).round(1),
    "pct_active": (100 * m.groupby("cluster_kmeans")["status"].
                   apply(lambda s: s.eq("Approved").mean())).round(1),
    "pct_bonus_abuser": (100 * m.groupby("cluster_kmeans")["is_bonus_abuser"].
                         apply(lambda s: s.eq(True).mean())).round(2),
    "pct_self_excluded": (100 * m.groupby("cluster_kmeans")["ever_self_excluded"].
                          apply(lambda s: s.eq(True).mean())).round(2),
    "pct_rg_med_high": (100 * m.groupby("cluster_kmeans")["rg_risk_rating"].
                        apply(lambda s: s.isin(["Medium", "High"]).mean())).round(2),
    "pct_aml_not_low": (100 * m.groupby("cluster_kmeans")["aml_risk_rating"].
                        apply(lambda s: s.isin(["Moderate", "High", "Extreme"]).mean())).round(2),
})
prof.to_csv(f"{CSV}/kmeans_cluster_numeric_profiles.csv")
display(prof)

,players,share_pct,med_tenure_days,med_deposits,med_withdrawals,med_bets,med_logins,med_net,med_win_rate_pct,med_monthly_deposits,med_ttfd_days,med_age,pct_male,pct_active,pct_bonus_abuser,pct_self_excluded,pct_rg_med_high,pct_aml_not_low
cluster_kmeans,,,,,,,,,,,,,,,,,,
0,9078,15.1,101.0,280.0,50.0,1541.0,52.0,125.0,83.7,89.0,-1.0,38.0,54.3,90.2,5.41,1.92,0.26,0.76
1,13366,22.3,184.0,20.0,42.0,27.0,3.0,-12.0,76.9,7.0,-1.0,29.0,63.1,89.3,15.56,0.33,0.05,0.17
2,10787,18.0,331.0,8180.0,5328.0,45351.0,413.0,2121.0,93.0,1018.0,-1.0,40.0,53.2,79.4,4.75,6.95,11.11,4.73
3,15122,25.2,208.0,45.0,0.0,195.0,4.0,40.0,65.4,9.0,-1.0,37.0,54.7,83.3,1.28,4.56,1.19,0.57
4,8125,13.6,179.0,435.0,670.0,1875.0,85.0,-150.0,100.4,114.0,-1.0,33.0,60.9,88.2,16.87,3.42,2.01,1.16
5,3444,5.7,369.0,596.0,248.0,3196.0,114.5,145.0,88.7,56.0,74.0,38.0,53.3,85.0,5.92,4.07,2.67,1.66


In [3]:
# ---- categorical / risk / acquisition profile per cluster (share of cluster) ----
dims = {
    "status": ["Approved", "Pending", "Closed", "Duplicated", "Suspended", "SelfExclusion", "Timeout"],
    "traffic_source": ["Organic", "Google", "Referred", "Facebook", "Referred From Paid Traffic",
                       "Liftoff", "Moloco"],
    "operational_profit_segment": ["LowValue", "MediumValue", "HighValue", "Vip1", "Vip2", "Vip3",
                                   "Toddler", "NegativeValue"],
    "registration_platform": ["Web", "iOS App", "Android App"],
    "gender": ["Male", "Female", "Other"],
    "rg_risk_rating": ["NoRisk", "Low", "Medium", "High"],
    "aml_risk_rating": ["Low", "Moderate", "High", "Extreme"],
}
rows = []
for dim, cats in dims.items():
    ct = pd.crosstab(m["cluster_kmeans"], m[dim], normalize="index") * 100
    for cl in range(6):
        for cat in cats:
            if cat in ct.columns:
                rows.append({"cluster": cl, "dimension": dim, "category": cat,
                             "pct_within_cluster": round(float(ct.loc[cl, cat]), 3)})
catp = pd.DataFrame(rows).query("pct_within_cluster >= 0.05")
catp.to_csv(f"{CSV}/kmeans_cluster_categorical_profiles.csv", index=False)

flags = pd.DataFrame({
    "cluster": range(6),
    "pct_test_account": (100 * m.groupby("cluster_kmeans")["is_test_account"].
                         apply(lambda s: s.eq(True).mean())).round(3).values,
    "pct_pep": (100 * m.groupby("cluster_kmeans")["is_pep"].mean()).round(3).values,
    "pct_sanctions": (100 * m.groupby("cluster_kmeans")["has_sanctions"].mean()).round(3).values,
    "pct_mfa": (100 * m.groupby("cluster_kmeans")["has_mfa_enabled"].
                apply(lambda s: s.eq(True).mean())).round(2).values,
})
flags.to_csv(f"{CSV}/kmeans_cluster_flags.csv", index=False)
display(flags)
print("saved categorical profiles + risk flags")

,cluster,pct_test_account,pct_pep,pct_sanctions,pct_mfa
0,0,0.000,0.000,0.0,1.92
1,1,0.000,0.000,0.0,1.28
2,2,0.000,0.009,0.0,2.96
3,3,0.000,0.000,0.0,1.01
4,4,0.012,0.000,0.0,2.28
5,5,0.000,0.000,0.0,3.57


saved categorical profiles + risk flags


In [4]:
# ---- archetype naming (K-Means) ----
ARCH = pd.DataFrame([
    {"cluster": 2, "archetype": "High-Value VIP / Whales",
     "reasoning": "long tenure, highest deposits/bets/logins, strongly net-positive"},
    {"cluster": 4, "archetype": "High-Spend Net-Loss Gamers",
     "reasoning": "active and withdrawal-heavy, deeply net-negative, bonus-abuse-prone"},
    {"cluster": 0, "archetype": "Steady Balanced Regulars",
     "reasoning": "mid engagement, net-positive but modest, clean risk profile"},
    {"cluster": 3, "archetype": "Casual Low-Stakes Winners",
     "reasoning": "small deposits, near-zero withdrawals, lowest bonus abuse, low activity"},
    {"cluster": 5, "archetype": "Loyal Slow-Converters",
     "reasoning": "longest tenure, very delayed first deposit, moderate-high engagement"},
    {"cluster": 1, "archetype": "Dormant / Lapsed Light Accounts",
     "reasoning": "minimal activity, short-lived or unused, bonus-chasing, young"},
])
prof0 = pd.read_csv(f"{CSV}/kmeans_cluster_numeric_profiles.csv")
tab = pd.DataFrame({"cluster": prof0["cluster_kmeans"] if "cluster_kmeans" in prof0 else range(6),
                    "players": prof0["players"], "share_pct": prof0["share_pct"]})
tab = tab.merge(ARCH[["cluster", "archetype"]], on="cluster", how="right").sort_values("cluster")
tab.to_csv(f"{CSV}/kmeans_cluster_archetypes.csv", index=False)
display(tab)
print("kmeans profiling complete")

,cluster,players,share_pct,archetype
2,0,9078,15.1,Steady Balanced Regulars
5,1,13366,22.3,Dormant / Lapsed Light Accounts
0,2,10787,18.0,High-Value VIP / Whales
3,3,15122,25.2,Casual Low-Stakes Winners
1,4,8125,13.6,High-Spend Net-Loss Gamers
4,5,3444,5.7,Loyal Slow-Converters


kmeans profiling complete
